In [ ]:
# Submodule smoke test: import the package + walk every submodule.
import importlib
import pkgutil

import typer

print(f"typer.__version__ = {getattr(typer, '__version__', 'unknown')}")
print(f"typer.__path__ = {typer.__path__}")

_SKIP = frozenset({"typer.__main__", "typer._click._winconsole"})

failures = []
for module_info in pkgutil.walk_packages(typer.__path__, prefix='typer.'):
    if module_info.name in _SKIP:
        continue
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        failures.append((module_info.name, type(e).__name__, str(e)[:200]))

if failures:
    for name, exc_type, msg in failures:
        print(f"FAIL {name}: {exc_type}: {msg}")
    raise AssertionError(f"{len(failures)} submodule(s) of typer failed to import")
print("\nAll submodules import cleanly")

In [ ]:
# Example 2: Use typer.Argument and typer.Option with type conversion.
import typer
from typer.testing import CliRunner

app = typer.Typer()

@app.command()
def add(
    x: int = typer.Argument(...),
    y: int = typer.Argument(...),
    verbose: bool = typer.Option(False, "--verbose", "-v"),
):
    total = x + y
    if verbose:
        typer.echo(f"{x} + {y} = {total}")
    else:
        typer.echo(str(total))

runner = CliRunner()
result = runner.invoke(app, ["3", "4", "-v"])
print("stdout:", result.stdout)
assert result.exit_code == 0
assert "3 + 4 = 7" in result.stdout


In [ ]:
# Example 3: Multiple commands and the --help output.
# NOTE: invoked in a SUBPROCESS, not via CliRunner — inside a Jupyter kernel rich's
# console detects Jupyter and routes typer's rich-formatted help through IPython
# display, so CliRunner's captured stdout comes back empty (kernel-only artifact,
# not a typer defect). A real subprocess exercises the true end-user help path.
import pathlib, subprocess, sys, tempfile

SCRIPT = """
import typer
app = typer.Typer(help="Demo app with multiple commands")

@app.command()
def create(name: str):
    typer.echo(f"created {name}")

@app.command()
def delete(name: str):
    typer.echo(f"deleted {name}")

if __name__ == "__main__":
    app()
"""

with tempfile.TemporaryDirectory() as td:
    script_path = pathlib.Path(td) / "demo_cli.py"
    script_path.write_text(SCRIPT)
    result = subprocess.run([sys.executable, str(script_path), "--help"],
                            capture_output=True, text=True, timeout=120)
print(result.stdout[:400])
assert result.returncode == 0
assert "create" in result.stdout
assert "delete" in result.stdout


In [ ]:
# Example 4: Sub-apps via add_typer for nested command groups.
import typer
from typer.testing import CliRunner

app = typer.Typer()
items_app = typer.Typer()
app.add_typer(items_app, name="items")

@items_app.command("list")
def items_list():
    typer.echo("item1,item2,item3")

@items_app.command("show")
def items_show(item_id: int):
    typer.echo(f"item={item_id}")

runner = CliRunner()
result = runner.invoke(app, ["items", "show", "42"])
print("stdout:", result.stdout)
assert result.exit_code == 0
assert "item=42" in result.stdout


In [ ]:
# Example 5: Error handling — typer.Exit and parameter validation.
import typer
from typer.testing import CliRunner

app = typer.Typer()

@app.command()
def divide(x: float, y: float):
    if y == 0:
        typer.echo("cannot divide by zero", err=True)
        raise typer.Exit(code=2)
    typer.echo(str(x / y))

runner = CliRunner()
ok = runner.invoke(app, ["10", "4"])
assert ok.exit_code == 0
assert "2.5" in ok.stdout

fail = runner.invoke(app, ["1", "0"])
print("exit_code:", fail.exit_code)
assert fail.exit_code == 2

bad = runner.invoke(app, ["not-a-number", "1"])
assert bad.exit_code != 0
print("All assertions passed")
